# 03 — LSM correlation pairs (P_LUX 2017–March 2020)

Estimates the 6 correlations of liver stiffness (LSM) against the rest of the matrix:

- LSM ↔ {BMI, LDL-C, SBP, FPG, smoking, eGFR}

**LSM is only measured in the 2017 – March 2020 pre-pandemic combined release** (the first NHANES cycle with FibroScan elastography; the variable is `LUXSMED`). This notebook merges `P_LUX` with `P_DEMO` (already cached) and pulls the other risks from the cycle's `P_BMX`, `P_BPX`, `P_TRIGLY`, `P_GLU`, `P_BIOPRO`, `P_SMQ` files.

Time-trend stratification is **not** possible (single combined cycle); age stratification is reported as a stability check.

In [1]:
import os, urllib.request, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

DATA = Path(os.path.abspath(os.path.join('..', 'data')))
RAW_PP = DATA / 'raw' / 'nhanes' / '2017_2020_prepandemic'
DERIVED = DATA / 'derived'
OUT = Path('outputs'); OUT.mkdir(parents=True, exist_ok=True)
RAW_PP.mkdir(parents=True, exist_ok=True)

## 1. Download (or load cached) P_LUX cycle XPT files

Files: P_DEMO, P_LUX, P_BMX, P_BPXO (oscillometric BP), P_TRIGLY, P_GLU, P_BIOPRO, P_SMQ. All from the NHANES 2017-2018 page (the pre-pandemic combined release uses this URL prefix). 'P_BPXO' is the oscillometric BP file used in the pre-pandemic release in lieu of the older auscultatory P_BPX.

In [2]:
FILES = [
    'P_DEMO.xpt', 'P_LUX.xpt', 'P_BMX.xpt', 'P_BPXO.xpt',
    'P_TRIGLY.xpt', 'P_GLU.xpt', 'P_BIOPRO.xpt', 'P_SMQ.xpt',
]
for f in FILES:
    out = RAW_PP / f
    if out.exists():
        print(f'cached: {out.name} ({out.stat().st_size:,} bytes)')
        continue
    url = f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{f}'
    print(f'downloading {f} ...')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as r:
        out.write_bytes(r.read())
    print(f'  saved {out.stat().st_size:,} bytes')

cached: P_DEMO.xpt (3,614,720 bytes)
cached: P_LUX.xpt (1,105,920 bytes)
cached: P_BMX.xpt (2,520,640 bytes)
cached: P_BPXO.xpt (1,039,840 bytes)
cached: P_TRIGLY.xpt (409,360 bytes)
cached: P_GLU.xpt (164,160 bytes)
cached: P_BIOPRO.xpt (3,420,640 bytes)
cached: P_SMQ.xpt (1,428,560 bytes)


## 2. Parse + merge by `SEQN`

In [3]:
DEMO_COLS  = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA']
LUX_COLS   = ['SEQN', 'LUXSMED', 'LUAXSTAT']
BMX_COLS   = ['SEQN', 'BMXBMI']
# Oscillometric BP: BPXOSY1, BPXOSY2, BPXOSY3 — average across valid readings
BPX_COLS   = ['SEQN', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3']
TRIG_COLS  = ['SEQN', 'LBDLDL']
GLU_COLS   = ['SEQN', 'LBXGLU', 'WTSAFPRP']  # fasting subsample weight
BIO_COLS   = ['SEQN', 'LBXSCR']
SMQ_COLS   = ['SEQN', 'SMQ020', 'SMQ040']

def load_xpt(name, cols):
    df = pd.read_sas(RAW_PP / name)
    keep = [c for c in cols if c in df.columns]
    return df[keep]

demo = load_xpt('P_DEMO.xpt', DEMO_COLS)
lux  = load_xpt('P_LUX.xpt',  LUX_COLS)
bmx  = load_xpt('P_BMX.xpt',  BMX_COLS)
bpx  = load_xpt('P_BPXO.xpt', BPX_COLS)
trig = load_xpt('P_TRIGLY.xpt', TRIG_COLS)
glu  = load_xpt('P_GLU.xpt',  GLU_COLS)
bio  = load_xpt('P_BIOPRO.xpt', BIO_COLS)
smq  = load_xpt('P_SMQ.xpt',  SMQ_COLS)

# Mean SBP across the 3 oscillometric readings (NHANES analytic guideline:
# average all valid readings, drop the first if 4 are present — but only 3
# are taken here)
bpx['SBP_MEAN'] = bpx[['BPXOSY1', 'BPXOSY2', 'BPXOSY3']].mean(axis=1)
bpx = bpx[['SEQN', 'SBP_MEAN']]

for f in [demo, lux, bmx, bpx, trig, glu, bio, smq]:
    f['SEQN'] = f['SEQN'].astype(int)

df = (demo
      .merge(lux,  on='SEQN', how='left')
      .merge(bmx,  on='SEQN', how='left')
      .merge(bpx,  on='SEQN', how='left')
      .merge(trig, on='SEQN', how='left')
      .merge(glu,  on='SEQN', how='left')
      .merge(bio,  on='SEQN', how='left')
      .merge(smq,  on='SEQN', how='left'))
print(f'merged: {len(df):,} rows from P_DEMO')
print(f'  with LSM (LUXSMED) measured: {df["LUXSMED"].notna().sum():,}')

merged: 15,560 rows from P_DEMO
  with LSM (LUXSMED) measured: 9,700


In [4]:
df['sex'] = df['RIAGENDR'].map({1.0: 'Male', 2.0: 'Female'})
df['age_years'] = df['RIDAGEYR'].astype(float)
df['exam_complete'] = df['LUAXSTAT'] == 1.0
df['LSM_KPA'] = df['LUXSMED']

def smoking_cat(row):
    if row['SMQ020'] == 2.0:
        return 3
    if row['SMQ020'] == 1.0:
        if row['SMQ040'] in (1.0, 2.0):
            return 1
        if row['SMQ040'] == 3.0:
            return 2
    return np.nan
df['smoking_cat'] = df.apply(smoking_cat, axis=1)

def ckd_epi_2021(scr, age, female):
    if pd.isna(scr) or pd.isna(age) or pd.isna(female):
        return np.nan
    kappa = 0.7 if female else 0.9
    alpha = -0.241 if female else -0.302
    sex_factor = 1.012 if female else 1.0
    ratio = scr / kappa
    return 142 * (min(ratio, 1) ** alpha) * (max(ratio, 1) ** -1.200) * (0.9938 ** age) * sex_factor
df['eGFR'] = df.apply(
    lambda r: ckd_epi_2021(r['LBXSCR'], r['age_years'], r['sex'] == 'Female'), axis=1
)

# Restrict to MEC-examined adults with a valid LSM exam
ana = df[df['exam_complete']
          & df['LSM_KPA'].notna()
          & df['WTMECPRP'].fillna(0).gt(0)
          & (df['age_years'] >= 20)].copy()
trial = ana[(ana['age_years'] >= 65) & (ana['age_years'] <= 80)].copy()
print(f'MEC-examined adults 20+ with LSM:                 n = {len(ana):,}')
print(f'  with all 6 risks non-null:                      n = {ana[["BMXBMI", "LBDLDL", "SBP_MEAN", "LBXGLU", "smoking_cat", "eGFR"]].notna().all(axis=1).sum():,}')
print(f'Trial-band 65-80 with LSM:                       n = {len(trial):,}')

MEC-examined adults 20+ with LSM:                 n = 7,396
  with all 6 risks non-null:                      n = 3,138
Trial-band 65-80 with LSM:                       n = 1,787


## 3. Weighted Spearman + paired-PSU jackknife (continuous-NHANES variance design `SDMVSTRA` × `SDMVPSU`)

In [5]:
def weighted_rank(x, w):
    o = np.argsort(x, kind='stable')
    cw = np.cumsum(w[o])
    r = np.empty_like(x, dtype=float)
    r[o] = cw - w[o] / 2.0
    return r

def w_spearman(x, y, w):
    rx, ry = weighted_rank(x, w), weighted_rank(y, w)
    mx, my = np.average(rx, weights=w), np.average(ry, weights=w)
    cov = np.average((rx - mx) * (ry - my), weights=w)
    sx = np.sqrt(np.average((rx - mx) ** 2, weights=w))
    sy = np.sqrt(np.average((ry - my) ** 2, weights=w))
    if sx == 0 or sy == 0: return np.nan
    return float(cov / (sx * sy))

def paired_jackknife_spearman(df, x_col, y_col,
                              wt='WTMECPRP', psu='SDMVPSU', stratum='SDMVSTRA'):
    sub = df[[x_col, y_col, wt, psu, stratum]].dropna()
    sub = sub[sub[wt] > 0]
    if len(sub) < 30:
        return np.nan, np.nan, len(sub)
    x = sub[x_col].values.astype(float)
    y = sub[y_col].values.astype(float)
    w = sub[wt].values.astype(float)
    s = sub[stratum].astype(int).values
    p = sub[psu].astype(int).values
    theta = w_spearman(x, y, w)
    var_sum = 0.0
    for st in np.unique(s):
        in_str = s == st
        psus_in = np.unique(p[in_str])
        if len(psus_in) < 2:
            continue
        for psu_id in psus_in:
            wr = w.copy()
            mask_in = in_str & (p == psu_id)
            mask_other = in_str & (p != psu_id)
            wr[mask_in] = 0.0
            wr[mask_other] *= 2.0
            if wr.sum() <= 0:
                continue
            t = w_spearman(x, y, wr)
            if not np.isnan(t):
                var_sum += (t - theta) ** 2
    return theta, np.sqrt(var_sum), len(sub)

## 4. Sign convention + headline 6 pairs (trial band 65–80)

Same convention as notebooks 01 / 02. LSM (`LSM_KPA`): higher = worse fibrosis, no flip.

In [6]:
trial['smoking_signed'] = 4 - trial['smoking_cat']
trial['eGFR_signed']    = -trial['eGFR']
ana['smoking_signed']   = 4 - ana['smoking_cat']
ana['eGFR_signed']      = -ana['eGFR']

OTHER_RISKS = {
    'BMI':                'BMXBMI',
    'LDL_C':              'LBDLDL',
    'SBP':                'SBP_MEAN',
    'FPG':                'LBXGLU',
    'smoking':            'smoking_signed',
    'kidney_dysfunction': 'eGFR_signed',
}
rows = []
for label, col in OTHER_RISKS.items():
    rho, se, n = paired_jackknife_spearman(trial, 'LSM_KPA', col)
    rows.append({
        'pair': f'liver_stiffness ↔ {label}',
        'risk_a': 'liver_stiffness',
        'risk_b': label,
        'n': n,
        'rho': rho,
        'se':  se,
        'lo':  rho - 1.96 * se if not pd.isna(se) else np.nan,
        'hi':  rho + 1.96 * se if not pd.isna(se) else np.nan,
    })
lsm_headline = pd.DataFrame(rows).round(3)
print('Trial-band 65-80, weighted Spearman, 95 % CI from paired-PSU jackknife:')
print(lsm_headline[['pair', 'n', 'rho', 'se', 'lo', 'hi']].to_string(index=False))

Trial-band 65-80, weighted Spearman, 95 % CI from paired-PSU jackknife:
                                pair    n    rho    se     lo     hi
               liver_stiffness ↔ BMI 1752  0.213 0.055  0.106  0.321
             liver_stiffness ↔ LDL_C  826 -0.226 0.067 -0.357 -0.096
               liver_stiffness ↔ SBP 1608  0.032 0.053 -0.072  0.135
               liver_stiffness ↔ FPG  842  0.228 0.067  0.097  0.359
           liver_stiffness ↔ smoking 1787  0.010 0.054 -0.096  0.116
liver_stiffness ↔ kidney_dysfunction 1650  0.098 0.048  0.005  0.191


## 5. Age stratification — point estimates only

(No time-trend stratification: P_LUX is a single combined cycle.)

In [7]:
AGE_BANDS = {'40-64': (40, 65), '65-80': (65, 81), '80+': (80, 200)}
rows = []
for label, col in OTHER_RISKS.items():
    row = {'pair': f'liver_stiffness ↔ {label}'}
    for ag, (lo, hi) in AGE_BANDS.items():
        sub = ana[ana['age_years'].between(lo, hi - 1)]
        rho, _, n = paired_jackknife_spearman(sub, 'LSM_KPA', col)
        row[f'n_{ag}'] = n
        row[ag] = rho
    rows.append(row)
lsm_age = pd.DataFrame(rows)
lsm_age['range'] = (lsm_age[list(AGE_BANDS.keys())].max(axis=1)
                     - lsm_age[list(AGE_BANDS.keys())].min(axis=1))
cols = ['pair'] + sum([['n_'+k, k] for k in AGE_BANDS.keys()], []) + ['range']
print('LSM pair correlations by age band:')
print(lsm_age[cols].round(3).to_string(index=False))
diverging = lsm_age[lsm_age['range'] > 0.10]
print(f'\n{len(diverging)} of {len(lsm_age)} pairs diverge by > 0.10 across age bands.')

LSM pair correlations by age band:
                                pair  n_40-64  40-64  n_65-80  65-80  n_80+    80+  range
               liver_stiffness ↔ BMI     3327  0.339     1752  0.213    392  0.068  0.271
             liver_stiffness ↔ LDL_C     1588 -0.095      826 -0.226    175 -0.298  0.203
               liver_stiffness ↔ SBP     3094  0.157     1608  0.032    341  0.084  0.126
               liver_stiffness ↔ FPG     1634  0.188      842  0.228    179  0.108  0.120
           liver_stiffness ↔ smoking     3345  0.064     1787  0.010    410  0.003  0.062
liver_stiffness ↔ kidney_dysfunction     3151  0.028     1650  0.098    384  0.085  0.070

4 of 6 pairs diverge by > 0.10 across age bands.


In [8]:
lsm_headline.to_parquet(OUT / 'lsm_headline.parquet')
lsm_age.to_parquet(OUT / 'lsm_age_strat.parquet')
print(f'wrote {OUT}/lsm_*.parquet')

wrote outputs/lsm_*.parquet
